In [1]:
%matplotlib inline
import csv
import random
import time
from collections import Counter, deque
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from PIL import Image
from ultralytics import YOLO

In [2]:
DATA = Path("C:/Users/Krish/Downloads/archive/data")
WEIGHTS = "../weights/best.pt"
RUNS = Path("../runs")
RUNS.mkdir(exist_ok=True)

CLASSES = ["smoke", "fire"]
COLOURS = {"smoke": (160, 160, 160), "fire": (0, 0, 255)}

TRAIN = False
MODELS = ["yolov8n.pt"]
EPOCHS = 50

THRESHOLD = 0.5
WINDOW = 10
MIN_HITS = 5

with open(RUNS / "data.yaml", "w") as f:
    yaml.safe_dump({"path": str(DATA), "train": "train/images", "val": "val/images",
                    "test": "test/images", "names": {0: "smoke", 1: "fire"}}, f)

In [3]:
for split in ["train", "val", "test"]:
    images = [p for p in (DATA / split / "images").iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    boxes = Counter()
    images_with = Counter()
    background = 0
    missing = 0

    for img in images:
        label = DATA / split / "labels" / (img.stem + ".txt")
        if not label.exists():
            missing += 1
            ids = []
        else:
            ids = [int(line.split()[0]) for line in label.read_text().splitlines() if line.strip()]
        if not ids:
            background += 1
        boxes.update(ids)
        images_with.update(set(ids))

    print(f"[{split}] {len(images)} images, {background} background, {missing} missing labels")
    for c, name in enumerate(CLASSES):
        print(f"    {name:<6} {boxes[c]:>6} boxes in {images_with[c]:>6} images")

[train] 14122 images, 6458 background, 0 missing labels
    smoke    7794 boxes in   6894 images
    fire     9638 boxes in   3828 images
[val] 3099 images, 1375 background, 0 missing labels
    smoke    1756 boxes in   1550 images
    fire     2176 boxes in    879 images
[test] 4306 images, 2005 background, 0 missing labels
    smoke    2315 boxes in   2081 images
    fire     2878 boxes in   1115 images


In [4]:
CHECK_CORRUPT = False

if CHECK_CORRUPT:
    for split in ["train", "val", "test"]:
        bad = []
        for img in (DATA / split / "images").iterdir():
            try:
                with Image.open(img) as im:
                    im.load()
            except Exception:
                bad.append(img.name)
        print(split, len(bad), bad[:10])